# V7 / qllm math tutorial

Teaches **PyTorch tensor mechanics** and the **linear algebra** used in the Phase-Associative Memory LM ([`v7/model.py`](../v7/model.py)), aligned with [`v7/EXPERIMENTS_V7.md`](../v7/EXPERIMENTS_V7.md).

**Run this notebook from the repository root** (so `import v7.model` works), or ensure the repo root is on `PYTHONPATH`.

**Dependencies:** project uses `torch`. For Jupyter: `pip install jupyter ipykernel` or install the optional extra: `uv sync --extra notebook` (see `pyproject.toml`).

**Coming from Node or PHP?** Start at **§0** — nested arrays, row-major memory, and a **foldable JSON** view of the same `(B,T,D,2)` tensor.


## Data flow (training)

```mermaid
flowchart LR
  subgraph split [SplitReal]
    z["z[...,0] z[...,1]"]
  end
  subgraph pam [PAM_training]
    QKV["Q K V projections"]
    Dmat["D from cumsum gamma"]
    W["W = Q K* scaled"]
    Y["Y = (W*D) @ V"]
  end
  z --> QKV
  QKV --> W
  QKV --> Dmat
  Dmat --> Y
  W --> Y
```


In [5]:
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

# Repo root: cwd or parent of notebooks/
_here = Path.cwd().resolve()
REPO = None
for p in (_here, _here.parent):
    if (p / "v7" / "model.py").exists():
        REPO = p
        break
if REPO is None:
    raise RuntimeError("Could not find v7/model.py — open Jupyter from repo root or set cwd.")
sys.path.insert(0, str(REPO))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device, "| repo:", REPO)


device: cuda | repo: /home/gowrav/Development/qllm2


## 0. From JavaScript / PHP → tensor shapes

If you know **nested arrays**, you already know 90% of indexing.

### Same idea as nested arrays

Shape `(B, T, D, 2)` means: **`B` batches → each has `T` time steps → each step has `D` numbers → each number is a pair `[real, imag]`**.

**JavaScript mental model** (same outer-to-inner order as PyTorch):

```js
// z[b][t][d] is length-2 array: [real, imag]
const z = [];
for (let b = 0; b < B; b++) {
  z[b] = [];
  for (let t = 0; t < T; t++) {
    z[b][t] = [];
    for (let d = 0; d < D; d++) {
      z[b][t][d] = [real, imag];
    }
  }
}
// PyTorch: z[b, t, d, 0] is real, z[b, t, d, 1] is imag
```

**PHP**: `$z[$b][$t][$d]` is a two-element vector (real, imag). PyTorch uses **comma indices** `z[b, t, d, :]` instead of chained `[b][t][d]`.

### Row-major (C order) — not “columns first”

PyTorch/NumPy store data in **row-major** order: the **last** index changes fastest in memory (like C arrays). You do **not** need matrix “column vs row” intuition for indexing — you need **which bracket is outer vs inner**. Left = outer = larger structure.

### `dim` and negative indices

| In `(B, T, D, 2)` | Name (typical in LM code) | Positive `dim` | Negative `dim` |
|-------------------|---------------------------|----------------|----------------|
| Batch             | independent sequences     | `0`            | `-4`           |
| Time              | token position            | `1`            | `-3`           |
| Feature           | embedding dimension       | `2`            | `-2`           |
| Complex pair      | real / imag               | `3`            | `-1`           |

`dim=-1` always means the **last** axis (here: real vs imag). `mean(dim=-1)` averages over that axis; `mean(dim=-2)` averages over `D` (features).

### Foldable tree in the next cell

The next cell defines helpers and shows an **`IPython.display.JSON`** view: in Jupyter you can **click to expand** batch → time → feature → real/imag, like folding JSON in the browser.

In [ ]:
import json
from typing import Any, Dict, List

try:
    from IPython.display import JSON, display as ipy_display
except ImportError:
    JSON = None  # type: ignore
    ipy_display = None  # type: ignore


def show_json_tree(data: Dict[str, Any]) -> None:
    """Foldable JSON in Jupyter; falls back to print if IPython is not installed."""
    if JSON is not None and ipy_display is not None:
        ipy_display(JSON(data))
    else:
        print(json.dumps(data, indent=2))


def tensor_to_labeled_tree(x: torch.Tensor, dim_names: List[str]) -> Dict[str, Any]:
    """Nested dict: batch=0 → time=1 → feature=2 → {re, im}. Expand/collapse in Jupyter JSON view."""
    x = x.detach().cpu().contiguous()
    if len(dim_names) != x.ndim:
        raise ValueError(f"dim_names length {len(dim_names)} != ndim {x.ndim}")

    def walk(idx: tuple) -> Any:
        depth = len(idx)
        if depth == x.ndim - 1 and x.shape[-1] == 2:
            return {
                "re": round(float(x[idx + (0,)].item()), 5),
                "im": round(float(x[idx + (1,)].item()), 5),
            }
        if depth == x.ndim:
            return round(float(x[idx].item()), 5)
        axis = dim_names[depth]
        n = x.shape[depth]
        return {f"{axis}={i}": walk(idx + (i,)) for i in range(n)}

    return walk(())


def print_axis_cheat_sheet(shape: tuple, names: List[str]) -> None:
    for i, (s, n) in enumerate(zip(shape, names)):
        neg = i - len(shape)
        print(f"  dim {i:2d} (dim {neg:3d}): size {s:4d}  ← {n}")


B_, T_, D_ = 2, 3, 4
torch.manual_seed(42)
demo = torch.randn(B_, T_, D_, 2, device=device)

names = ["batch", "time", "feature", "complex [re, im]"]
print("Shape:", tuple(demo.shape))
print_axis_cheat_sheet(tuple(demo.shape), names)
print("\n--- Expand/collapse in Jupyter (JSON view) ---")
show_json_tree(tensor_to_labeled_tree(demo, names))

print("\n--- Nested lists == JS z[b][t][d][0/1] ---")
js_like = demo.cpu().tolist()
text = json.dumps(js_like, indent=2)
print(text[:1500] + ("\n... (truncated)" if len(text) > 1500 else ""))

## 1. Shapes, indexing, `dim=-1`

(See **§0** for JS/PHP nested-array intuition and the foldable JSON tree.)

V7 stores complex numbers as **split real**: last axis is `[real, imag]`, shape `[..., dim, 2]`.

- `...` (ellipsis) means “all leading dimensions” (same as filling in `:` for every axis to the left).
- On a tensor with shape `(B, T, D, 2)`: `dim=-1` is the **complex pair**, `dim=-2` is **feature `D`**, `dim=-3` is **time `T`**, `dim=-4` is **batch**.


In [4]:
B, T, D = 2, 5, 8
z = torch.randn(B, T, D, 2, device=device)
assert z.shape == (B, T, D, 2)

# Peek one complex number: batch 0, time 0, feature 0  →  same as JS z[0][0][0]
print("z[0,0,0] as [re, im]:", z[0, 0, 0].tolist())

# Foldable tree for a corner of z (uses helpers from §0)
show_json_tree(
    tensor_to_labeled_tree(
        z[:2, :2, :3].cpu(),
        ["batch", "time", "feature", "complex [re, im]"],
    )
)

real = z[..., 0]   # drop last axis, keep real only → shape (B, T, D)
imag = z[..., 1]
assert real.shape == (B, T, D)

# On `real` with shape (B, T, D): last axis is D (features). mean(dim=-1) averages over D.
rms_core = (real.square() + imag.square()).mean(dim=-1, keepdim=True)
print("rms_core shape:", rms_core.shape, "  # (B, T, 1) — one RMS per (batch, time)")

# On full z (B, T, D, 2): dim=-1 is real vs imag — sum pairs into one number per (b,t,d)
print("z.sum(dim=-1) shape:", z.sum(dim=-1).shape, "  # (B, T, D)")


tensor([[[[ 2.5441, -0.7163],
          [-0.4934,  0.1267],
          [ 0.1014, -0.4035],
          [ 0.9023,  0.8099],
          [-0.6884,  0.1372],
          [ 1.0377,  0.0925],
          [-0.3752, -0.0908],
          [ 2.0639, -1.8164]],

         [[-0.2719,  0.2811],
          [-1.0399,  0.7765],
          [ 0.8814,  0.0444],
          [-1.4870,  1.1334],
          [ 1.3268, -1.2616],
          [ 0.9501, -0.6558],
          [ 0.9098, -0.6290],
          [-0.6587,  2.0811]],

         [[ 1.4151, -0.3091],
          [-0.2055,  2.0562],
          [-0.0490, -0.6361],
          [-0.5359, -0.1310],
          [-0.2945,  1.2275],
          [ 1.0549,  0.3576],
          [ 1.6378, -0.2310],
          [ 0.7883, -0.0807]],

         [[-0.3924,  1.2673],
          [ 1.0420, -0.4945],
          [-1.1637,  1.5740],
          [ 0.7116,  0.6104],
          [ 1.2852, -0.6533],
          [ 1.1171, -1.0067],
          [ 1.2912,  1.6028],
          [ 0.1332,  1.0703]],

         [[-1.1161, -0.8396],
  

## 2. `unsqueeze` and broadcasting

`cabs(z)` is `(B,T,D)`. To divide `z` by magnitude **per component**, we need `mag` to have shape `(B,T,D,1)` so it broadcasts against `(B,T,D,2)`.

That is what `.unsqueeze(-1)` does: insert a size-1 axis at the end.


In [ ]:
def cabs(x):
    return torch.sqrt(x[..., 0].square() + x[..., 1].square() + 1e-8)

mag = cabs(z)                      # (B, T, D)
phase = z / (mag.unsqueeze(-1) + 1e-8)
assert phase.shape == z.shape
# phase is unit magnitude in complex sense along (r,i)
print("mag shape:", mag.shape, "| phase shape:", phase.shape)


## 3. `view`, `contiguous`, `transpose`

- **`transpose(i, j)`** swaps two dimensions (may make storage non-contiguous).
- **`view`** requires a contiguous tensor — use **`.contiguous().view(...)`** after transpose (see PAM output in `model.py`).

**Fused QKV:** linear output is reshaped to `[B, T, 3, H, d, 2]`, then Q/K/V are taken and heads move to axis 1: `[B, H, T, d, 2]`.


In [ ]:
B, T, H, d = 2, 7, 3, 4
inner = H * d
qkv_flat = torch.randn(B, T, 3 * inner, 2, device=device)
qkv = qkv_flat.view(B, T, 3, H, d, 2)
q = qkv[:, :, 0].transpose(1, 2).contiguous()
k = qkv[:, :, 1].transpose(1, 2).contiguous()
v = qkv[:, :, 2].transpose(1, 2).contiguous()
assert q.shape == (B, H, T, d, 2)
print("q shape:", q.shape)

# view fails without contiguous — demonstrate
try:
    qkv[:, :, 0].transpose(1, 2).view(B, H, T * d * 2)
except RuntimeError as e:
    print("expected error:", e)
fixed = qkv[:, :, 0].transpose(1, 2).contiguous().view(B, H, T * d * 2)
print("contiguous view OK:", fixed.shape)


## 4. Batched matrix multiply `@`

For tensors with shape `(..., m, n) @ (..., n, p)` PyTorch applies one matmul per batch of leading dimensions `...`.

In PAM, keys are transposed on the last two axes: `kr.transpose(-1, -2)` has shape `[B, H, d, T]` so `qr @ kr.T` gives `[B, H, T, T]`.


In [ ]:
B, H, T, d = 2, 3, 11, 5
qr = torch.randn(B, H, T, d, device=device)
kr = torch.randn(B, H, T, d, device=device)
scores = qr @ kr.transpose(-1, -2)
assert scores.shape == (B, H, T, T)
print("scores:", scores.shape)

# Same as F.linear on last dim: y = x @ W.T
W = torch.randn(d, d, device=device)
x = torch.randn(B, T, d, device=device)
y_lin = F.linear(x, W)
y_mm = x @ W.T
assert torch.allclose(y_lin, y_mm)


## 5. Split-real complex: `cmul`, `cconj`

Same formulas as [`v7/model.py`](../v7/model.py). We compare a hand implementation to the library.


In [ ]:
from v7.model import cmul, cconj


def cmul_hand(a, b):
    return torch.stack(
        [
            a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1],
            a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0],
        ],
        dim=-1,
    )


a = torch.randn(2, 3, 4, 2, device=device)
b = torch.randn(2, 3, 4, 2, device=device)
assert torch.allclose(cmul_hand(a, b), cmul(a, b))

cj = cconj(a)
assert torch.allclose(cj[..., 0], a[..., 0]) and torch.allclose(cj[..., 1], -a[..., 1])
print("cmul / cconj OK")


## 6. Causal mask and `torch.tril`

Dual-form PAM only uses pairs `(t, i)` with `i <= t`. The code stores a lower-triangular mask of ones and multiplies invalid positions by a large negative before `exp` (see `_dual_form_block`).


In [ ]:
T = 5
causal = torch.tril(torch.ones(T, T, device=device))
print(causal)
off = torch.ones(T, T, device=device) * (-1e4)
masked = causal + (1 - causal) * off
print("masked[0,4] (should be large negative):", masked[0, 4].item())


## 7. Decay matrix `D` from `gamma`

Per head and batch, `gamma[b,h,t]` is in `(0,1]`. Cumulative log decay `C[t] = sum_{j<=t} -log gamma_j`.

Then `log D[t,i] = C[i] - C[t]` (for `i <= t`) gives the log of the product `gamma_{i+1} * ... * gamma_t` — the factor that scales how much position `i` contributes when writing output at `t`.

The code uses `unsqueeze(-1)` and `unsqueeze(-2)` to form all `(t,i)` pairs, then `transpose(-1,-2)` to align axes with `W`.


In [ ]:
def build_D_from_gamma(gamma_1d: torch.Tensor) -> torch.Tensor:
    """gamma_1d: (T,) — returns (T, T) lower-triangular D as in V7."""
    T = gamma_1d.shape[0]
    log_gamma = torch.log(gamma_1d + 1e-6)
    C = torch.cumsum(-log_gamma, dim=-1)
    log_D = (C.unsqueeze(-1) - C.unsqueeze(-2)).transpose(-1, -2)
    causal = torch.tril(torch.ones(T, T, device=gamma_1d.device))
    log_D = log_D * causal + (1 - causal) * (-1e4)
    return torch.exp(log_D.clamp(max=0.0))


T = 4
g = torch.tensor([0.9, 0.8, 0.95, 0.85], device=device)
D = build_D_from_gamma(g)
# Manual: D[t,i] = prod_{j=i+1}^t g[j], i<=t
for t in range(T):
    for i in range(T):
        if i > t:
            continue
        prod = torch.ones((), device=device)
        for j in range(i + 1, t + 1):
            prod = prod * g[j]
        assert torch.allclose(D[t, i], prod, atol=1e-5), (t, i, D[t, i], prod)
print("D matches manual product along diagonals.")
print(D)


## 8. Mini PAM: recurrent vs dual form

**Recurrent:** `S_t = gamma_t S_{t-1} + v_t ⊗ k_t^*`, then `y_t = S_t q_t` (complex).

**Dual (training):** build `W = Q K^*` (complex), mask with `D`, multiply by `V`.

**Shapes in the outer product:** for `v_prime[:, :, t]` with shape `(B, H, d, 2)`, `unsqueeze(-2)` yields `(B, H, d, 1, 2)` (column-like). Conjugate keys use `unsqueeze(-3)` → `(B, H, 1, d, 2)` (row-like). Broadcasting multiplies to `(B, H, d, d, 2)` — one `d×d` complex matrix per head.

We use `PhaseAssociativeLayer._dual_form_block` from the repo and a recurrent loop that mirrors `v7/model.py`.


In [ ]:
from v7.model import PhaseAssociativeLayer


def pam_recurrent(q, k, v_prime, gamma, d: int):
    """Match v7/model.py recurrent branch for training comparison. Shapes: q,k,v_prime (1,1,T,d,2); gamma (1,1,T)."""
    B, H, T = q.shape[0], q.shape[1], q.shape[2]
    scale = d ** -0.5
    S = torch.zeros(B, H, d, d, 2, device=q.device, dtype=q.dtype)
    y_list = []
    for t in range(T):
        v_t = v_prime[:, :, t].unsqueeze(-2)
        k_t = k[:, :, t]
        k_conj = torch.stack([k_t[..., 0], -k_t[..., 1]], dim=-1).unsqueeze(-3)
        outer_r = v_t[..., 0] * k_conj[..., 0] - v_t[..., 1] * k_conj[..., 1]
        outer_i = v_t[..., 0] * k_conj[..., 1] + v_t[..., 1] * k_conj[..., 0]
        outer = torch.stack([outer_r, outer_i], dim=-1)
        g = gamma[:, :, t].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        S = S * g + outer
        q_t = q[:, :, t].unsqueeze(-3) * scale
        sq_r = S[..., 0] * q_t[..., 0] - S[..., 1] * q_t[..., 1]
        sq_i = S[..., 0] * q_t[..., 1] + S[..., 1] * q_t[..., 0]
        y_list.append(torch.stack([sq_r, sq_i], dim=-1).sum(dim=-2))
    return torch.stack(y_list, dim=2)


T, d = 6, 4
B = H = 1
q = torch.randn(B, H, T, d, 2, device=device)
k = torch.randn(B, H, T, d, 2, device=device)
v_prime = torch.randn(B, H, T, d, 2, device=device)
gamma = torch.sigmoid(torch.randn(B, H, T, device=device) * 0.2 + 0.5)

scale = d ** -0.5
q_s = q * scale
causal = torch.tril(torch.ones(T, T, device=device))
y_dual, _ = PhaseAssociativeLayer._dual_form_block(q_s, k, v_prime, gamma, causal)
y_rec = pam_recurrent(q, k, v_prime, gamma, d)
assert torch.allclose(y_dual, y_rec, atol=1e-4, rtol=1e-4)
print("dual form matches recurrent:", y_dual.shape)


## 9. Complex RoPE cache

`build_rope_cache` returns `[max_len, head_dim, 2]` as `(cos θ, sin θ)` for `e^{iθ}`. Apply with `cmul(q, pos)` so each frequency band rotates by position.


In [ ]:
from v7.model import build_rope_cache, cmul

max_len, head_dim = 32, 8
rope = build_rope_cache(max_len, head_dim).to(device=device)
assert rope.shape == (max_len, head_dim, 2)

T_use = 10
q = torch.randn(1, 2, T_use, head_dim, 2, device=device)
pos = rope[:T_use].to(dtype=q.dtype)
# broadcast pos to (1,1,T,d,2) then cmul
pos_b = pos.unsqueeze(0).unsqueeze(0)
q_rot = cmul(q, pos_b)
print("rotated q shape:", q_rot.shape)


## 10. Tied complex LM head

Logits: `Re(z · conj(embed))` for each vocab token. With split embeddings `E_r, E_i`:

`logits = z_r @ E_r^T + z_i @ E_i^T` (see `V7LM.forward`).


In [ ]:
B, T, D, V = 2, 5, 16, 100
z = torch.randn(B, T, D, 2, device=device)
E_r = torch.randn(V, D, device=device)
E_i = torch.randn(V, D, device=device)
zr, zi = z[..., 0], z[..., 1]
logits = zr @ E_r.T + zi @ E_i.T
assert logits.shape == (B, T, V)
print("logits:", logits.shape)


## 11. Optional: full `V7LM` forward (tiny preset)

Smoke test: random token ids → logits shape `[B, T, vocab_size]`.


In [ ]:
from v7.model import V7LM, PRESETS

cfg = PRESETS["tiny"]
model = V7LM(cfg).to(device).eval()
B, T = 2, 17
ids = torch.randint(0, cfg.vocab_size, (B, T), device=device)
logits, states = model(ids)
assert logits.shape == (B, T, cfg.vocab_size)
assert len(states) == cfg.n_layers
print("logits:", tuple(logits.shape), "| num layer states:", len(states))
